In [21]:
pip install bs4

Note: you may need to restart the kernel to use updated packages.


In [22]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [23]:
url = 'https://coinmarketcap.com/'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36'
}

In [24]:
page = requests.get(url, headers=headers)
soup = BeautifulSoup(page.text, 'html.parser')

In [25]:
table = soup.find('table')
titles = table.find_all('th')
header_names = [title.text.strip() for title in titles if title.text.strip() != '']

In [26]:
df = pd.DataFrame(columns = header_names[:9]) 

In [27]:
rows = table.find_all('tr')
for row in rows[1:11]:  
    row_data = row.find_all('td')
    individual_row = [data.text.strip() for data in row_data if data.text.strip() != '']
    if len(individual_row) >= len(df.columns):
        length = len(df)
        df.loc[length] = individual_row[:len(df.columns)]

In [28]:
print(df)

   #          Name       Price   1h %  24h %   7d %                Market Cap  \
0  1    BitcoinBTC  $74,193.46  0.71%  0.34%  2.83%  $1.48T$1,484,528,344,539   
1  2   EthereumETH   $2,317.09  0.99%  1.03%  4.86%  $279.42B$279,419,348,185   
2  3    TetherUSDT       $1.00  0.00%  0.01%  0.01%  $185.77B$185,772,401,653   
3  4        XRPXRP       $1.41  0.66%  3.05%  5.12%    $87.34B$87,337,162,041   
4  5        BNBBNB     $625.07  0.01%  0.80%  3.31%    $84.23B$84,230,214,935   
5  6      USDCUSDC     $0.9998  0.00%  0.02%  0.01%    $78.63B$78,632,218,116   
6  7     SolanaSOL      $85.77  0.95%  1.57%  3.04%    $49.34B$49,344,549,700   
7  8       TRONTRX     $0.3267  0.23%  0.11%  2.43%    $30.97B$30,966,373,312   
8  9  DogecoinDOGE    $0.09660  0.48%  2.47%  4.62%    $16.38B$16,381,200,195   

               Volume(24h) Circulating Supply  
0   $41,498,289,652559.54K         20.01M BTC  
1     $19,925,687,4868.60M        120.69M ETH  
2  $128,501,774,767128.48B       185.74B USDT

In [29]:
table = soup.find('table', class_='cmc-table')
rows = table.find('tbody').find_all('tr')

In [31]:
data = []

for row in rows[:100]:
    cells = row.find_all('td')
    if len(cells) < 9: continue  # Skip incomplete rows
    
    # 1. Rank (#)
    rank = cells[1].text.strip()
    
    # 2. Name & Symbol (Splitting nested <p> tags)
    name_info = cells[2].find_all('p')
    name = name_info[0].text if name_info else "N/A"
    symbol = name_info[1].text if len(name_info) > 1 else "N/A"
    
    # 3. Price (Targeting the span to avoid extra text)
    price = cells[3].text.strip()
    
    # 4. Percent Changes (1h, 24h, 7d)
    # Using a helper to check if the value is up or down based on icon class
    def get_pct(cell):
        val = cell.text.strip()
        icon = cell.find('span', class_='icon-Caret-up')
        return f"▲{val}" if icon else f"▼{val}"

    pct_1h = get_pct(cells[4])
    pct_24h = get_pct(cells[5])
    pct_7d = get_pct(cells[6])
    
    # 5. Market Cap (Improved)
    # We look for a span that contains a '$' sign, or just grab the first span text
    mc_span = cells[7].find('span', class_=lambda x: x != 'sc-66133f36-2') # Skip the tooltip icon
    if mc_span:
        market_cap = mc_span.text.strip()
    else:
        # Fallback: Get the last span or paragraph which usually holds the value
        market_cap = cells[7].find_all(['span', 'p'])[-1].text.strip()
    
    # 6. Volume (24h)
    # Usually the first <p> tag or the cell text itself
    vol_element = cells[8].find('p')
    volume = vol_element.text.strip() if vol_element else cells[8].text.strip()
    
    # 7. Circulating Supply
    # Usually the first <p> tag
    supply_element = cells[9].find('p')
    supply = supply_element.text.strip() if supply_element else cells[9].text.strip()
    
    data.append({
        'Rank': rank,
        'Name': name,
        'Symbol': symbol,
        'Price': price,
        '1h %': pct_1h,
        '24h %': pct_24h,
        '7d %': pct_7d,
        'Market Cap': market_cap,
        'Volume(24h)': volume,
        'Circulating Supply': supply
    })

In [32]:
# Create DataFrame
df = pd.DataFrame(data)

# Display result
print(df.to_string(index=False))

Rank                       Name Symbol      Price   1h %  24h %    7d % Market Cap      Volume(24h) Circulating Supply
     CoinMarketCap 20 Index DTF  CMC20    $152.02 ▼0.11% ▲0.68%  ▲3.53%    $15.98M       $2,121,556      105.13K CMC20
   1                    Bitcoin    BTC $74,193.46 ▼0.71% ▲0.34%  ▲2.83%     $1.48T  $41,498,289,652         20.01M BTC
   2                   Ethereum    ETH  $2,317.09 ▼0.99% ▼1.03%  ▲4.86%   $279.42B  $19,925,687,486        120.69M ETH
   3                     Tether   USDT      $1.00 ▲0.00% ▲0.01%  ▲0.01%   $185.77B $128,501,774,767       185.74B USDT
   4                        XRP    XRP      $1.41 ▼0.66% ▲3.05%  ▲5.12%    $87.34B   $3,921,838,611         61.56B XRP
   5                        BNB    BNB    $625.07 ▼0.01% ▲0.80%  ▲3.31%    $84.23B   $1,707,762,393        134.78M BNB
   6                       USDC   USDC    $0.9998 ▲0.00% ▲0.02%  ▼0.01%    $78.63B  $50,674,965,555        78.64B USDC
   7                     Solana    SOL     $85.7

In [36]:
df.to_csv('crypto_top.csv', index=False)
print("\nData saved to crypto.csv")


Data saved to crypto.csv


In [35]:
pip install selenium webdriver-manager

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.6 MB 7.3 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.6 MB 7.4 MB/s eta 0:00:01
   ------------------ --------------------- 4.5/9.6 MB 8.1 MB/s eta 0:00:01
   --------------------------- ------------ 6.6/9.6 MB 8.6 MB/s eta 0:00:01
   ------------------------------------ --- 8.7/9.6 MB 8.8 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 8.4 MB/s eta 0:00:00

  Attempting uninstall: urllib3

   ---- ----------------------------------- 1/9 [urllib3]
    Found existing installation: urllib3 2.3.0
   ---- ----------------------------------- 1/9 [urllib3]
    Uninstalling urllib3-2.3.0:
   ---- ----------------------------------- 1/9 [urllib3]
      Successfully uninstalled urllib3-2.3.0
   ---- ----------------------------------- 1/9 [urllib3]
   ---- ----------------------------------- 1/9 [urllib3]
   ---- -------------------

In [39]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

In [40]:
options = webdriver.ChromeOptions()
options.add_argument('--headless') # Runs without opening a window
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = 'https://coinmarketcap.com/'
driver.get(url)

for i in range(5):
    driver.execute_script("window.scrollBy(0, 2000);")
    time.sleep(1) # Give it a second to load the new rows

# 3. Grab the fully loaded HTML
soup = BeautifulSoup(driver.page_source, 'html.parser')
driver.quit()

table = soup.find('table', class_='cmc-table')
rows = table.find('tbody').find_all('tr')

print(f"Total rows found: {len(rows)}") # Should now be 100

Total rows found: 101


In [41]:
# This tells pandas to show every single row and column
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df)

   Rank                        Name Symbol       Price    1h %   24h %  \
0        CoinMarketCap 20 Index DTF  CMC20     $152.02  ▼0.11%  ▲0.68%   
1     1                     Bitcoin    BTC  $74,193.46  ▼0.71%  ▲0.34%   
2     2                    Ethereum    ETH   $2,317.09  ▼0.99%  ▼1.03%   
3     3                      Tether   USDT       $1.00  ▲0.00%  ▲0.01%   
4     4                         XRP    XRP       $1.41  ▼0.66%  ▲3.05%   
5     5                         BNB    BNB     $625.07  ▼0.01%  ▲0.80%   
6     6                        USDC   USDC     $0.9998  ▲0.00%  ▲0.02%   
7     7                      Solana    SOL      $85.77  ▼0.95%  ▲1.57%   
8     8                        TRON    TRX     $0.3267  ▲0.23%  ▼0.11%   
9     9                    Dogecoin   DOGE    $0.09660  ▼0.48%  ▲2.47%   
10   10                 Hyperliquid   HYPE      $43.86  ▼1.93%  ▼1.63%   
11   11                UNUS SED LEO    LEO      $10.14  ▼0.01%  ▼0.09%   
12   12                     Cardano   

In [47]:
import pandas as pd

# (Add this after your driver.quit() line)

data = []

# Loop through the rows found (skipping the header if necessary)
for row in rows:
    cells = row.find_all('td')
    if len(cells) < 10: 
        continue # Skip rows that don't have enough columns (like ads)
    
    # helper for Name/Symbol because they are in the same cell
    name_p = cells[2].find_all('p')
    name = name_p[0].text if name_p else "N/A"
    symbol = name_p[1].text if len(name_p) > 1 else "N/A"

    data.append({
        'Rank': cells[1].text.strip(),
        'Name': name,
        'Symbol': symbol,
        'Price': cells[3].text.strip(),
        '1h %': cells[4].text.strip(),
        '24h %': cells[5].text.strip(),
        '7d %': cells[6].text.strip(),
        'Market Cap': cells[7].text.strip(),
        'Volume(24h)': cells[8].text.strip(),
    })

# Create the DataFrame
df = pd.DataFrame(data)

In [48]:
print(df)

    Rank                                   Name   Symbol       Price   1h %   24h %      7d %                Market Cap              Volume(24h)
0                    CoinMarketCap 20 Index DTF    CMC20     $151.82  0.23%   0.46%     3.03%        $15.96M$15,958,365         $2,122,44313.97K
1      1                                Bitcoin      BTC  $74,392.78  0.07%   0.54%     2.78%  $1.49T$1,488,114,795,756   $41,689,822,230560.77K
2      2                               Ethereum      ETH   $2,327.04  0.17%   0.71%     4.75%  $280.84B$280,839,444,430     $20,147,320,0638.65M
3      3                                 Tether     USDT       $1.00  0.01%   0.01%     0.00%  $185.75B$185,750,525,196  $129,842,831,322129.83B
4      4                                    XRP      XRP       $1.42  0.09%   3.52%     4.89%    $87.78B$87,783,822,423      $3,964,236,4412.78B
5      5                                    BNB      BNB     $627.08  0.16%   1.07%     3.10%    $84.52B$84,522,380,346      $1,72

In [49]:
# 1. Force everything to stay on one line
pd.set_option('display.max_columns', None)      # Show all columns
pd.set_option('display.expand_frame_repr', False) # Prevent wrapping to a new line
pd.set_option('display.max_rows', None)         # Show all 100+ rows
pd.set_option('display.width', 2000)            # Give it a very wide workspace

# 2. Print the dataframe
print(df)

    Rank                                   Name   Symbol       Price   1h %   24h %      7d %                Market Cap              Volume(24h)
0                    CoinMarketCap 20 Index DTF    CMC20     $151.82  0.23%   0.46%     3.03%        $15.96M$15,958,365         $2,122,44313.97K
1      1                                Bitcoin      BTC  $74,392.78  0.07%   0.54%     2.78%  $1.49T$1,488,114,795,756   $41,689,822,230560.77K
2      2                               Ethereum      ETH   $2,327.04  0.17%   0.71%     4.75%  $280.84B$280,839,444,430     $20,147,320,0638.65M
3      3                                 Tether     USDT       $1.00  0.01%   0.01%     0.00%  $185.75B$185,750,525,196  $129,842,831,322129.83B
4      4                                    XRP      XRP       $1.42  0.09%   3.52%     4.89%    $87.78B$87,783,822,423      $3,964,236,4412.78B
5      5                                    BNB      BNB     $627.08  0.16%   1.07%     3.10%    $84.52B$84,522,380,346      $1,72

In [50]:
df.to_csv('crypto_top.csv', index=False)
print("\nData saved to crypto.csv")


Data saved to crypto.csv
